# Emotion Detection from Text — Training & Evaluation

**Project:** Emotion Detection from Text (tweet-level, 13 emotion classes)
**Model:** TF-IDF features + a linear classifier (logistic regression trained epoch-by-epoch)

This notebook is an **orchestration notebook**: all real logic (data loading, cleaning, splitting, model definition, training loop, evaluation) lives in the `src/` package. This notebook only imports those functions, sets configuration, and calls them in order.

**Sections:**
1. Imports & configuration
2. Load the dataset
3. Data sanity check (before cleaning)
4. Explore the dataset
5. Clean the text
6. Data sanity check (after cleaning)
7. Train / validation / test split
8. Build the model (vectorizer + classifier)
9. Train and validate (epoch loop)
10. Training curves
11. Final evaluation on the test set
12. Save the trained model
13. Try the model on your own sentences

## 1. Imports & Configuration

`src/config.py` holds every file path and hyperparameter used below. Change values there, not in this notebook.

In [ ]:
from src import config
from src.data_loader import load_dataset, get_class_counts
from src.data_sanity_check import run_sanity_checks
from src.text_cleaner import clean_dataframe
from src.data_splitter import split_data
from src.model_builder import build_vectorizer, build_classifier, combine_into_pipeline
from src.training_pipeline import train_and_validate
from src.evaluator import plot_training_curves, evaluate_model, plot_confusion_matrix
from src.model_io import save_model
from src.predictor import predict_emotion

## 2. Load the Dataset

In [ ]:
df = load_dataset(config.DATA_PATH)
df.head()

## 3. Data Sanity Check (Raw Data)

Before doing anything else, check the raw data for missing values, missing labels, and duplicate rows.

In [ ]:
_ = run_sanity_checks(df, text_column=config.TEXT_COLUMN, label_column=config.LABEL_COLUMN)

## 4. Explore the Dataset

How many tweets exist for each emotion? Some emotions (like `anger` and `boredom`) have very few examples — that class imbalance is expected, but it means the model will find those emotions harder to learn.

In [ ]:
class_counts = get_class_counts(df, label_column=config.LABEL_COLUMN)
class_counts

In [ ]:
class_counts.plot(kind="bar", figsize=(10, 4), title="Number of Tweets per Emotion")

## 5. Clean the Text

Raw tweets contain links, @mentions, hashtags, and punctuation. `clean_dataframe()` cleans all of that and stores the result in a new column.

In [ ]:
df = clean_dataframe(df, text_column=config.TEXT_COLUMN, new_column=config.CLEAN_TEXT_COLUMN)
df[[config.TEXT_COLUMN, config.CLEAN_TEXT_COLUMN, config.LABEL_COLUMN]].head()

## 6. Data Sanity Check (Cleaned Data)

Run the sanity check again, this time also checking whether any tweets became an empty string after cleaning (e.g. a tweet that was just a URL).

In [ ]:
_ = run_sanity_checks(
    df,
    text_column=config.TEXT_COLUMN,
    label_column=config.LABEL_COLUMN,
    clean_text_column=config.CLEAN_TEXT_COLUMN,
)

## 7. Train / Validation / Test Split

- **Train** — the model learns from this.
- **Validation** — checked after every training epoch to track learning progress.
- **Test** — untouched until the very end, for a final unbiased evaluation.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df,
    text_column=config.CLEAN_TEXT_COLUMN,
    label_column=config.LABEL_COLUMN,
    val_size=config.VAL_SIZE,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

## 8. Build the Model

Two untrained pieces: a TF-IDF vectorizer (text -> numbers) and a linear classifier (numbers -> emotion).

In [ ]:
vectorizer = build_vectorizer(max_features=config.MAX_FEATURES, ngram_range=config.NGRAM_RANGE)
classifier = build_classifier(
    alpha=config.ALPHA,
    learning_rate=config.LEARNING_RATE,
    random_state=config.RANDOM_STATE,
)

## 9. Train and Validate

Trains for `config.EPOCHS` epochs. After every epoch we print train/validation accuracy and loss.

In [ ]:
classifier, vectorizer, history = train_and_validate(
    vectorizer, classifier,
    X_train, y_train,
    X_val, y_val,
    epochs=config.EPOCHS,
)

## 10. Training Curves

If validation accuracy flattens out (or validation loss starts rising) while train accuracy keeps improving, that's a sign of overfitting.

In [ ]:
plot_training_curves(history)

## 11. Final Evaluation on the Test Set

The test set was never touched during training or validation, so this is our best estimate of how the model performs on brand-new tweets.

In [ ]:
model = combine_into_pipeline(vectorizer, classifier)
y_pred = evaluate_model(model, X_test, y_test)

In [ ]:
emotion_labels = sorted(df[config.LABEL_COLUMN].unique())
plot_confusion_matrix(y_test, y_pred, labels=emotion_labels)

## 12. Save the Trained Model

In [ ]:
save_model(model, path=config.MODEL_SAVE_PATH)

## 13. Try the Model on Your Own Sentences

In [ ]:
my_sentences = [
    "I am so happy today, everything is going great!",
    "I can't believe this happened, I'm furious right now.",
    "I feel so alone and sad tonight.",
]

predictions = predict_emotion(model, my_sentences)

for sentence, emotion in zip(my_sentences, predictions):
    print(f"{emotion:12s} -> {sentence}")